In [7]:
import os
print(os.getcwd())
print(os.listdir('../data'))

/Users/chanee/Desktop/DataScience - Humor styles analysis - Rishi Khare/Notebooks
['codebook.txt', 'data.csv', 'clean_humor_data.csv']


In [8]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/clean_humor_data.csv')

# Remove age outliers
df_clean = df[df['age'] <= 90].copy()

# Create age group column
df_clean['age_group'] = np.where(df_clean['age'] < 23, 'Under 23', '23 or above')

print(df_clean.shape)
print(df_clean['age_group'].value_counts())

(989, 40)
age_group
Under 23       495
23 or above    494
Name: count, dtype: int64


In [9]:
from scipy import stats

for age_group in ['Under 23', '23 or above']:
    subset = df_clean[df_clean['age_group'] == age_group]
    corr, p = stats.spearmanr(subset['selfdefeating'], subset['selfenhancing'])
    print(f"--- {age_group} ---")
    print(f"Spearman r: {corr:.4f}")
    print(f"P-value:    {p:.4f}")
    print()

--- Under 23 ---
Spearman r: 0.2641
P-value:    0.0000

--- 23 or above ---
Spearman r: 0.1362
P-value:    0.0024



## Spearman Correlation Analysis

Spearman correlation was used to measure the strength of association between self-defeating and self-enhancing humor for each age group separately. Spearman was chosen over Pearson because the data is ordinal, rated on a 1-5 scale, and does not assume a normal distribution. An alpha level of 0.05 was used for all tests.
For respondents under 23, the Spearman correlation was r = 0.2641 with a p-value of 0.0000. This indicates a weak but statistically significant positive association between self-defeating and self-enhancing humor in younger respondents. For respondents aged 23 or above, the correlation was r = 0.1362 with a p-value of 0.0024. This also indicates a statistically significant positive association, though weaker than the younger group.
In both cases, the direction of the correlation is positive — meaning higher self-defeating humor scores are associated with higher self-enhancing humor scores, not lower as originally theorized.

In [10]:
def fishers_z_test(r1, r2, n1, n2):
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    se = np.sqrt(1/(n1-3) + 1/(n2-3))
    z_stat = (z1 - z2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    return z_stat, p_value

n1 = len(df_clean[df_clean['age_group'] == 'Under 23'])
n2 = len(df_clean[df_clean['age_group'] == '23 or above'])

z_stat, p_value = fishers_z_test(0.2641, 0.1362, n1, n2)

print(f"Z statistic: {z_stat:.4f}")
print(f"P-value:     {p_value:.4f}")
if p_value < 0.05:
    print("Result: The difference between correlations is statistically significant.")
else:
    print("Result: The difference is NOT statistically significant.")

Z statistic: 2.0922
P-value:     0.0364
Result: The difference between correlations is statistically significant.


## Fisher's Z Test analysis

Fisher's Z test was conducted to determine whether the difference between the two correlation coefficients (r = 0.2641 for under 23 vs r = 0.1362 for 23 or above) was statistically significant. The test produced a Z statistic of 2.0922 and a p-value of 0.0364. Since 0.0364 is below the alpha level of 0.05, the difference between the two correlations is statistically significant. This means the association between self-defeating and self-enhancing humor is meaningfully stronger in younger respondents than in older respondents.

## Overall Finding

The data does not support the original hypothesis that self-defeating humor replaces self-enhancing humor. Instead, both humor styles show a statistically significant positive association in both age groups, meaning they tend to coexist rather than compete. However, the strength of this association differs significantly by age group — younger respondents show a stronger link between the two styles than older respondents. This suggests that as people age, the relationship between self-defeating and self-enhancing humor becomes weaker, possibly reflecting greater emotional differentiation or maturity in humor use.